# Combined parallel blend from scratch

This notebook demonstrates the combined pipeline: score every candidate with a GBT-style scorer and an LLM-context scorer, normalize both score streams, then blend them with `final_score = 0.8 * gbt_score + 0.2 * llm_context_score`.

In [1]:
from math import log2

user = {"likes": {"coffee", "brunch", "vietnamese"}, "price": 2, "max_distance": 4.0}
candidates = [
    {"id": "b1", "name": "Morning Bean", "tags": {"coffee", "bakery", "quiet"}, "price": 2, "distance": 0.7, "rating": 4.7, "reviews": 410, "label": 3},
    {"id": "b2", "name": "Pho Corner", "tags": {"vietnamese", "noodles", "casual"}, "price": 1, "distance": 2.2, "rating": 4.5, "reviews": 260, "label": 3},
    {"id": "b3", "name": "Quiet Study Cafe", "tags": {"coffee", "dessert", "quiet"}, "price": 2, "distance": 1.1, "rating": 4.1, "reviews": 95, "label": 2},
    {"id": "b4", "name": "Late Night BBQ", "tags": {"bbq", "korean", "loud"}, "price": 3, "distance": 6.5, "rating": 4.6, "reviews": 520, "label": 0},
    {"id": "b5", "name": "Campus Banh Mi", "tags": {"vietnamese", "sandwich", "cheap"}, "price": 1, "distance": 0.4, "rating": 4.2, "reviews": 140, "label": 2},
    {"id": "b6", "name": "Weekend Brunch Lab", "tags": {"brunch", "coffee", "popular"}, "price": 2, "distance": 3.2, "rating": 4.4, "reviews": 180, "label": 3},
]


In [2]:
def gbt_score(user, item):
    category = len(user["likes"] & item["tags"]) / len(user["likes"])
    price = 1 - min(abs(user["price"] - item["price"]), 3) / 3
    distance = max(0, 1 - item["distance"] / user["max_distance"])
    popularity = min(item["reviews"], 500) / 500
    return 0.40 * category + 0.20 * price + 0.20 * distance + 0.12 * (item["rating"] / 5) + 0.08 * popularity

def llm_context_score(user, item):
    score = 0.0
    score += 0.35 if "quiet" in item["tags"] else 0.0
    score += 0.30 if user["likes"] & item["tags"] else 0.0
    score += 0.20 if item["distance"] <= 1.5 else 0.05
    score += 0.10 if item["price"] <= 2 else -0.10
    score -= 0.25 if "loud" in item["tags"] else 0.0
    return score

def minmax(values):
    lo, hi = min(values), max(values)
    return [(value - lo) / (hi - lo) if hi > lo else 0.0 for value in values]


In [3]:
gbt_raw = [gbt_score(user, item) for item in candidates]
llm_raw = [llm_context_score(user, item) for item in candidates]
gbt_norm = minmax(gbt_raw)
llm_norm = minmax(llm_raw)

alpha, beta = 0.8, 0.2
ranked = []
for item, gbt, llm in zip(candidates, gbt_norm, llm_norm):
    ranked.append({**item, "gbt_score": round(gbt, 4), "llm_context_score": round(llm, 4), "final_score": round(alpha * gbt + beta * llm, 4)})
ranked = sorted(ranked, key=lambda row: row["final_score"], reverse=True)

[(row["id"], row["name"], row["gbt_score"], row["llm_context_score"], row["final_score"], row["label"]) for row in ranked]


[('b1', 'Morning Bean', 1.0, 1.0, 1.0, 3),
 ('b6', 'Weekend Brunch Lab', 0.899, 0.6, 0.8392, 3),
 ('b3', 'Quiet Study Cafe', 0.7598, 1.0, 0.8078, 2),
 ('b5', 'Campus Banh Mi', 0.6973, 0.72, 0.7018, 2),
 ('b2', 'Pho Corner', 0.5171, 0.6, 0.5337, 3),
 ('b4', 'Late Night BBQ', 0.0, 0.0, 0.0, 0)]

In [4]:
def dcg(labels):
    return sum((2 ** label - 1) / log2(rank + 2) for rank, label in enumerate(labels))

def ndcg(rows, k):
    actual = [row["label"] for row in rows[:k]]
    ideal = sorted((row["label"] for row in rows), reverse=True)[:k]
    return dcg(actual) / dcg(ideal) if dcg(ideal) else 0.0

print("combined ndcg@3", round(ndcg(ranked, 3), 4))
print("combined ndcg@5", round(ndcg(ranked, 5), 4))
print("combined ndcg@10", round(ndcg(ranked, 10), 4))


combined ndcg@3 0.8659
combined ndcg@5 0.9739
combined ndcg@10 0.9739
